<a href="https://colab.research.google.com/github/Chaman4211/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Chaman4211/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding 1 — the learned ranking has stronger top-of-list results than the rule baseline.** The research materials report average precision of about **0.468** and Precision@50 of **0.240** for baseline rules, compared with about **0.618** and **0.740** for the random-forest ranking on the anonymized starter slice. This is a useful measured comparison, not proof that the model will work equally well on every client or future time period.

**Methodology question:** Could we document exactly how the decline label was made and confirm that every feature was available before the label window? I would also like to see the comparison repeated on the same client-held-out rows and to know whether the reported gap is stable across different held-out clients. That would help readers understand how much the measured lift depends on this split.

**Finding 2 — the task is defined using an observed trend label, not a direct measure of successful recovery after a refresh.** In this project, the decline label is `trend_direction == "down"`, and `trend_direction` is derived from `trend_pct`. That makes it a useful proxy for studying an observed trend, but it is not the same as proving a page will decline in the future or improve after an edit.

**Methodology question:** Could the paper show the precise time window and threshold used to assign the label, and check whether the drop persists rather than reflecting noise, seasonality, or related pages sharing demand? For a future-forecasting claim, could features be frozen before a later outcome window and evaluated with a time-aware split? These checks would make the scope of the finding clearer, not invalidate the work.

I treat these as questions for stronger reporting, not as a grade of the paper.

In [1]:
# Small evidence table for the two findings discussed above.
# These are reported reference results from the research materials, not metrics from my Week-5 model.
import pandas as pd

paper_results = pd.DataFrame([
    {"reported_method": "Baseline rules", "average_precision": 0.468, "precision_at_50": 0.240},
    {"reported_method": "Random forest", "average_precision": 0.618, "precision_at_50": 0.740},
])
display(paper_results)
print("Reading note: reported results describe the anonymized starter slice and its evaluation design.")
print("They do not establish causal impact or guarantee future-period performance.")

,reported_method,average_precision,precision_at_50
0,Baseline rules,0.468,0.24
1,Random forest,0.618,0.74


Reading note: reported results describe the anonymized starter slice and its evaluation design.
They do not establish causal impact or guarantee future-period performance.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

For a before/after diagnostic, I compare the Week-5 Logistic Regression using a **random row split** with the same model using a **client-grouped split**. The random split is only a reference because pages from one client can appear in both sets. The grouped split keeps each client entirely in either training or test, which better tests performance on clients not seen during training. Both comparisons use the same features, label definition, preprocessing, and ranking metrics. The difference is an observed split-sensitivity check; it is not a time-based future forecast.

The code loads the anonymized starter CSV from the repository or its public starter copy, then prints the measured metrics and real error examples from the grouped holdout.

In [2]:
# Re-run the Week-5 Logistic Regression under random-row and client-grouped splits.
from pathlib import Path
import subprocess
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score

RANDOM_STATE = 42
DATA_FILE = "content_refresh_anonymized.csv"
cwd = Path.cwd().resolve()
search_roots = [cwd, *cwd.parents]
repo_root = next((p for p in search_roots if (p / "work" / "notebooks").is_dir()), None)
if repo_root is None:
    repo_root = cwd / "flyrank-ml-internship"
    if not repo_root.exists():
        subprocess.run(["git", "clone", "https://github.com/Chaman4211/flyrank-ml-internship.git", str(repo_root)], check=True)

csv_candidates = [repo_root / "data" / "raw" / DATA_FILE, cwd / "data" / "raw" / DATA_FILE, cwd / "flyrank-ml-internship-starter" / "data" / "raw" / DATA_FILE]
csv_path = next((p for p in csv_candidates if p.is_file() and p.stat().st_size > 0), None)
if csv_path is None:
    starter_dir = cwd / "flyrank-ml-internship-starter"
    if not starter_dir.exists():
        subprocess.run(["git", "clone", "https://github.com/Chaman4211/flyrank-ml-internship-starter.git", str(starter_dir)], check=True)
    candidate = starter_dir / "data" / "raw" / DATA_FILE
    if candidate.is_file() and candidate.stat().st_size > 0:
        csv_path = candidate
if csv_path is None:
    raise FileNotFoundError("Could not find content_refresh_anonymized.csv. Check that the repository or public starter data is available.")

df = pd.read_csv(csv_path, low_memory=False)
required = {"content_id", "client_id", "trend_direction", "days_since_last_update", "content_age_days", "word_count", "char_count", "search_volume", "competition", "cpc", "content_type", "main_intent", "competition_level"}
missing = sorted(required - set(df.columns))
if missing:
    raise ValueError(f"Required columns are missing: {missing}")

TARGET = "is_declining_label"
df[TARGET] = df["trend_direction"].eq("down").astype("int8")
NUMERIC_FEATURES = ["content_age_days", "days_since_last_update", "word_count", "char_count", "search_volume", "competition", "cpc"]
CATEGORICAL_FEATURES = ["content_type", "main_intent", "competition_level"]
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
y = df[TARGET].astype(int)
X = df[FEATURES].copy()
groups = df["client_id"].astype(str)

def make_model():
    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ("scaler", StandardScaler()),
    ])
    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="unknown")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])
    prep = ColumnTransformer([
        ("numeric", numeric_pipe, NUMERIC_FEATURES),
        ("categorical", categorical_pipe, CATEGORICAL_FEATURES),
    ])
    return Pipeline([
        ("preprocess", prep),
        ("model", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE)),
    ])

def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))[:min(k, len(y_true))]
    return float(np.asarray(y_true)[order].mean()) if len(order) else float("nan")

def evaluate_split(split_name, train_idx, test_idx):
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    if y_train.nunique() < 2 or y_test.nunique() < 2:
        raise ValueError(f"{split_name} split must contain both target classes in train and test.")
    model = make_model()
    model.fit(X.iloc[train_idx], y_train)
    probs = model.predict_proba(X.iloc[test_idx])[:, 1]
    return model, {
        "split": split_name,
        "train_rows": int(len(train_idx)),
        "test_rows": int(len(test_idx)),
        "test_positive_rate": float(y_test.mean()),
        "average_precision": float(average_precision_score(y_test, probs)),
        "precision_at_20": precision_at_k(y_test.to_numpy(), probs, 20),
        "precision_at_50": precision_at_k(y_test.to_numpy(), probs, 50),
    }, probs

random_train, random_test = train_test_split(np.arange(len(df)), test_size=0.25, random_state=RANDOM_STATE, stratify=y)
random_model, random_metrics, random_probs = evaluate_split("Before: random row split", random_train, random_test)

group_splitter = GroupShuffleSplit(n_splits=100, test_size=0.25, random_state=RANDOM_STATE)
group_train = group_test = None
for candidate_train, candidate_test in group_splitter.split(X, y, groups=groups):
    if y.iloc[candidate_train].nunique() == 2 and y.iloc[candidate_test].nunique() == 2:
        group_train, group_test = candidate_train, candidate_test
        break
if group_train is None:
    raise ValueError("Could not find a client-grouped split with both classes in train and test.")

model, grouped_metrics, grouped_probs = evaluate_split("After: client-grouped split", group_train, group_test)
train_clients = set(groups.iloc[group_train])
test_clients = set(groups.iloc[group_test])
assert train_clients.isdisjoint(test_clients), "Client overlap found between grouped train and test."
comparison = pd.DataFrame([random_metrics, grouped_metrics])
display(comparison.round(4))
print(f"Dataset: {len(df):,} rows × {df.shape[1]} columns; observed decline-label rate: {y.mean():.3f}")
print(f"Grouped split: {len(train_clients)} training clients, {len(test_clients)} test clients, overlap = {len(train_clients & test_clients)}")
print("Interpretation: compare the measured gap; do not assume the grouped result must be lower.")

# Real failure examples from the client-grouped test rows; no client names, URLs, or raw text are shown.
group_test_rows = df.iloc[group_test].copy()
group_y_test = y.iloc[group_test].to_numpy()
error_examples = group_test_rows[["days_since_last_update", "content_age_days", "word_count", "content_type"]].copy().reset_index(drop=True)
error_examples["actual_decline_label"] = group_y_test
error_examples["predicted_probability"] = grouped_probs
error_examples["predicted_label_at_0_50"] = (grouped_probs >= 0.50).astype(int)
error_examples["error_type"] = np.select([
    (error_examples["actual_decline_label"] == 0) & (error_examples["predicted_label_at_0_50"] == 1),
    (error_examples["actual_decline_label"] == 1) & (error_examples["predicted_label_at_0_50"] == 0),
], ["false_positive", "false_negative"], default="correct")
errors_only = error_examples.loc[error_examples["error_type"] != "correct"].copy()
print(f"Grouped test errors at threshold 0.50: {len(errors_only):,} of {len(error_examples):,}")
if errors_only.empty:
    print("No errors at threshold 0.50 in this split.")
else:
    print("Up to five real error examples (feature values only):")
    display(errors_only.head(5).assign(predicted_probability=lambda z: z["predicted_probability"].round(3)))

print("Limits: this is a grouped client holdout on a static snapshot, not a time-aware forecast or a causal test.")

,split,train_rows,test_rows,test_positive_rate,average_precision,precision_at_20,precision_at_50
0,Before: random row split,22500,7500,0.5420,0.6447,0.7,0.70
1,After: client-grouped split,22885,7115,0.5165,0.5315,0.6,0.64


Dataset: 30,000 rows × 45 columns; observed decline-label rate: 0.542
Grouped split: 24 training clients, 8 test clients, overlap = 0
Interpretation: compare the measured gap; do not assume the grouped result must be lower.
Grouped test errors at threshold 0.50: 3,330 of 7,115
Up to five real error examples (feature values only):


,days_since_last_update,content_age_days,word_count,content_type,actual_decline_label,predicted_probability,predicted_label_at_0_50,error_type
1,25,445,2481.0,keyword article,1,0.420,0,false_negative
3,103,238,1342.0,keyword article,0,0.657,1,false_positive
7,20,502,NaN,keyword article,1,0.317,0,false_negative
10,8,140,2542.0,comparison article,0,0.603,1,false_positive
11,20,187,2510.0,keyword article,0,0.576,1,false_positive


Limits: this is a grouped client holdout on a static snapshot, not a time-aware forecast or a causal test.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

The label is `trend_direction == "down"`, and that label source is not a feature. I exclude identifiers, `trend_direction`, `trend_pct`, direct last/previous-30-day trend inputs, current 90-day performance totals, and product-generated scores or flags. The model features are limited to age/freshness, content size, search-demand/competition context, and content categories.

A grouped split addresses repeated clients; it does **not** fix a feature whose time window overlaps the label window. Because the starter file is a snapshot rather than a timestamped prediction panel, I cannot demonstrate a future-only feature cutoff here. I therefore describe the result as ranking an observed snapshot label, not forecasting a future decline.

In [3]:
# Programmatic leakage checks for the exact feature list used above.
FORBIDDEN_FEATURES = {
    "content_id", "client_id", TARGET, "trend_direction", "trend_pct",
    "impressions_last_30d", "impressions_prev_30d",
    "clicks_last_30d", "clicks_prev_30d",
    "sessions_last_30d", "sessions_prev_30d",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d",
    "users_90d", "engaged_sessions_90d", "ai_sessions_90d",
    "health_score", "needs_ctr_fix", "is_quick_win",
}
feature_intersection = sorted(set(FEATURES) & FORBIDDEN_FEATURES)
assert not feature_intersection, f"Forbidden feature(s) entered model: {feature_intersection}"
assert set(FEATURES).issubset(df.columns), "A selected feature is missing from the dataset."
assert TARGET not in FEATURES and "trend_direction" not in FEATURES and "trend_pct" not in FEATURES
assert "content_id" not in FEATURES and "client_id" not in FEATURES
assert train_clients.isdisjoint(test_clients), "Grouped split has client overlap."

leakage_audit = pd.DataFrame([
    {"check": "Label excluded from features", "status": "PASS" if TARGET not in FEATURES else "FAIL"},
    {"check": "trend_direction and trend_pct excluded", "status": "PASS" if not ({"trend_direction", "trend_pct"} & set(FEATURES)) else "FAIL"},
    {"check": "IDs excluded from features", "status": "PASS" if not ({"content_id", "client_id"} & set(FEATURES)) else "FAIL"},
    {"check": "Direct 30-day trend inputs excluded", "status": "PASS" if not ({"impressions_last_30d", "impressions_prev_30d", "clicks_last_30d", "clicks_prev_30d", "sessions_last_30d", "sessions_prev_30d"} & set(FEATURES)) else "FAIL"},
    {"check": "Current 90-day traffic totals and product flags excluded", "status": "PASS" if not (FORBIDDEN_FEATURES & set(FEATURES)) else "FAIL"},
    {"check": "No client overlap in honest split", "status": "PASS" if train_clients.isdisjoint(test_clients) else "FAIL"},
])
display(leakage_audit)
assert leakage_audit["status"].eq("PASS").all()
print("Feature set:", ", ".join(FEATURES))
print("Time-window limitation: this snapshot does not prove every remaining feature was observed strictly before a future outcome window.")

,check,status
0,Label excluded from features,PASS
1,trend_direction and trend_pct excluded,PASS
2,IDs excluded from features,PASS
3,Direct 30-day trend inputs excluded,PASS
4,Current 90-day traffic totals and product flag...,PASS
5,No client overlap in honest split,PASS


Feature set: content_age_days, days_since_last_update, word_count, char_count, search_volume, competition, cpc, content_type, main_intent, competition_level
Time-window limitation: this snapshot does not prove every remaining feature was observed strictly before a future outcome window.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Too strong:** “My model predicts which pages will decline and tells us which pages need refreshing.”

**Safer claim:** “On this anonymized starter snapshot, I measured how a Logistic Regression ranking aligns with the observed decline label. I compared a random-row split with a client-grouped holdout and report the measured metrics, base rates, and error examples. The results are directional decision-support for human review; they do not establish future decline, explain why a page changed, or show that refreshing a page causes recovery.”

I will report the actual numbers printed by the notebook rather than choose the better-looking split. If grouped results differ from random-split results, that difference is part of the finding.

In [4]:
# Print the safe claim alongside the measured comparison produced in Section 2.
print("SAFE CLAIM")
print("On this anonymized starter snapshot, I measured how a Logistic Regression ranking aligns with the observed decline label.")
print("I compared a random-row split with a client-grouped holdout and report the measured metrics, base rates, and error examples.")
print("The results are directional decision-support for human review; they do not establish future decline, explain why a page changed, or show that refreshing a page causes recovery.")

required_safe_terms = {"observed", "measured", "directional", "decision-support"}
safe_claim = (
    "On this anonymized starter snapshot, I measured how a Logistic Regression ranking aligns with the observed decline label. "
    "The results are directional decision-support for human review."
).lower()
assert all(term in safe_claim for term in required_safe_terms), "Safe claim should include the agreed careful language."
print("\nClaim-language check: PASS")
print("Do not write a claim of future forecasting or causal impact from these results.")

SAFE CLAIM
On this anonymized starter snapshot, I measured how a Logistic Regression ranking aligns with the observed decline label.
I compared a random-row split with a client-grouped holdout and report the measured metrics, base rates, and error examples.
The results are directional decision-support for human review; they do not establish future decline, explain why a page changed, or show that refreshing a page causes recovery.

Claim-language check: PASS
Do not write a claim of future forecasting or causal impact from these results.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.